# Chapter 2 — Getting Familiar with Hugging Face Datasets

This notebook is the entry point for Chapter 2's text-classification workflow. Before training any model, it explores the `datasets` library and shows how a dataset from the Hugging Face Hub is represented in Python. The central object is `DatasetDict`, which stores named splits such as `train`, `validation`, and `test`. Each split behaves like a table: it has rows, columns, feature metadata, slicing support, and efficient access to individual fields.

The main dataset used in this chapter is the Emotion dataset. It contains short English messages labeled with six emotion classes. In later notebooks, these labels become the supervised targets for both classical machine-learning baselines and transformer fine-tuning. This notebook focuses only on data access: loading a public dataset, inspecting its schema, reading examples, and seeing how the same API can also load local CSV files.

Keep the outputs visible while studying this notebook. The printed `DatasetDict`, feature schema, examples, and slices are not decoration; they show how Hugging Face datasets organize text and labels before tokenization and modeling.

## Imports

We start with two Hugging Face utilities: `list_datasets()` for discovering datasets on the Hub and `load_dataset()` for loading a dataset into the local Python session.

In [17]:
# Imports for discovering datasets on the Hub and loading them locally.
from huggingface_hub import list_datasets
from datasets import load_dataset

## Listing one dataset from the Hub

This quick call proves that the notebook can communicate with the Hugging Face Hub. The variable name says `one_datasets`, and the print message says “first 10,” but the code intentionally requests only one item with `limit=1`.

In [18]:
# Request one dataset entry from the Hugging Face Hub and print the returned metadata.
one_datasets = list(list_datasets(limit= 1))

print(f"The first 10 datasets:\n{one_datasets}")

The first 10 datasets:
[DatasetInfo(id='facebook/research-plan-gen', author='facebook', sha='8ae1ba08759afd32b20eb959ea6addf25c2c0929', created_at=datetime.datetime(2025, 12, 28, 14, 3, 33, tzinfo=datetime.timezone.utc), last_modified=datetime.datetime(2026, 1, 2, 14, 56, 26, tzinfo=datetime.timezone.utc), private=False, gated=False, disabled=False, downloads=1104, downloads_all_time=None, likes=173, paperswithcode_id=None, tags=['size_categories:10K<n<100K', 'format:parquet', 'modality:text', 'library:datasets', 'library:pandas', 'library:polars', 'library:mlcroissant', 'arxiv:2512.23707', 'region:us'], trending_score=173, card_data=None, siblings=None, xet_enabled=None)]


## Loading the Emotion dataset

The `emotion` dataset is downloaded and cached locally. Hugging Face Datasets returns a structured object rather than a raw Pandas DataFrame, which is why later cells inspect the split structure.

In [20]:
# Load the Emotion dataset; this returns a DatasetDict with train/validation/test splits.
emotions = load_dataset("emotion")

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

## Inspecting the dataset container

Displaying `emotions` reveals the available splits and the columns stored in each split. This is the first sanity check before using the data for supervised learning.

In [21]:
# Display the full DatasetDict so we can inspect split names, columns, and row counts.
emotions

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

## Selecting the training split

A `DatasetDict` contains multiple splits. Here we isolate the `train` split because most following inspections focus on the data used for model fitting.

In [23]:
# Select only the training split for detailed inspection.
train_ds = emotions["train"]
train_ds

Dataset({
    features: ['text', 'label'],
    num_rows: 16000
})

## Counting training examples

The length of a split gives the number of rows/examples. This is useful for estimating training time, batch counts, and whether the dataset size is realistic for fine-tuning.

In [24]:
# Count the number of examples in the training split.
len(train_ds)

16000

## Checking column names

The Emotion dataset has two main fields: the raw text and the integer label. Knowing the column names is necessary before writing tokenizer and preprocessing functions.

In [25]:
# Show the names of the columns available in each row.
train_ds.column_names

['text', 'label']

## Inspecting feature metadata

The `features` object describes the type of each column. For classification, the label feature is especially useful because it stores the mapping between class IDs and human-readable class names.

In [26]:
# Inspect the feature schema, including the ClassLabel mapping for the label column.
train_ds.features

{'text': Value('string'),
 'label': ClassLabel(names=['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'])}

## Reading a single example

Indexing with an integer returns one row as a Python dictionary. This makes it easy to understand the exact format passed through the pipeline.

In [27]:
# Retrieve a single training example as a Python dictionary.
train_ds[0]

{'text': 'i didnt feel humiliated', 'label': 0}

## Slicing multiple examples

Slicing returns several rows at once. This is a quick way to preview the dataset without converting the entire split to Pandas.

In [28]:
# Slice the first five rows to preview multiple examples at once.
print(train_ds[:5])

{'text': ['i didnt feel humiliated', 'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake', 'im grabbing a minute to post i feel greedy wrong', 'i am ever feeling nostalgic about the fireplace i will know that it is still on the property', 'i am feeling grouchy'], 'label': [0, 0, 3, 2, 3]}


## Accessing one column directly

Column access retrieves values from a single field. Here, reading the first few texts helps us see the natural language content without the labels.

In [29]:
# Read only the text column for the first five examples.
print(train_ds["text"][:5])

['i didnt feel humiliated', 'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake', 'im grabbing a minute to post i feel greedy wrong', 'i am ever feeling nostalgic about the fireplace i will know that it is still on the property', 'i am feeling grouchy']


## Loading local CSV files with the same API

`load_dataset()` is not only for Hub datasets. The same function can wrap local CSV files into a `DatasetDict`, which is useful when moving from book examples to Kaggle or portfolio datasets.

In [40]:
# Load local CSV files with the same datasets API used for Hub datasets.
# This is useful when applying the chapter workflow to your own files.
data_path = '../../ml_projects/ml_modeling/kaggle_competitions/Disaster Twitts/data/train.csv'
test_path = '../../ml_projects/ml_modeling/kaggle_competitions/Disaster Twitts/data/test.csv'
dataset = load_dataset('csv', data_files= {'train': data_path, 'test': test_path})

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

## Inspecting the local CSV dataset

The output shows how the external train/test CSV files are represented after loading. This reinforces the idea that different data sources can share the same Hugging Face Dataset interface.

In [41]:
# Display the DatasetDict created from the local CSV files.
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'target'],
        num_rows: 7613
    })
    test: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'target'],
        num_rows: 3263
    })
})

## Empty working cell

This cell was present in the original notebook and is preserved for continuity. You can use it for small experiments while studying the chapter.

In [ ]:
# Empty scratch cell preserved from the original notebook.
